In [1]:
import sys
sys.path.append('..')
import selex_dca, utils

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

import utils
import selex_dca
import selex_distribution, energy_models, tree, data_loading, training, callback, sampling
import diagnostic, specialized_models

from matplotlib import cm
import pickle
import copy

import logomaker
import pandas as pd

/home/scrotti/Aptamer2025py/experiments/../selex_dca.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
experiment_id_idx = 0
experiment_ids = ['Dop8V030', 'Dop8V930', 'Dop8V2430']
experiment_id = experiment_ids[experiment_id_idx]
round_ids = ["ARN", "R01", "R02N"]

In [3]:
dtype = torch.float32
device = torch.device('cpu')

In [4]:
sequences, sequences_unique, counts, log_multinomial_factors = utils.sequences_counts_from_files(experiment_id, round_ids)

Extracting sequences...
Finished round ARN
Finished round R01
Finished round R02N


In [5]:
sequences_oh = [utils.one_hot(s) for s in sequences]
total_reads = torch.tensor([len(s) for s in sequences]).to(device)
L, q = sequences_oh[0][0].size()

In [6]:
from importlib import reload
reload(energy_models)
reload(selex_distribution)

<module 'selex_distribution' from '/home/scrotti/Aptamer2025py/experiments/../selex_distribution.py'>

In [29]:
class TransformerRegressor(torch.nn.Module):
    def __init__(self, seq_len, d_model=64, nhead=4, num_layers=1):
        super().__init__()
        
        # Project input features to d_model
        self.input_proj = torch.nn.Linear(1, d_model)
        
        # Transformer encoder (attention + FFN, no decoder needed)
        encoder_layer = torch.nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = torch.nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Collapse sequence + project to scalar output
        self.output_head = torch.nn.Linear(d_model * seq_len, 1)

    def forward(self, x):
        # x: (batch_size, seq_len, input_dim)
        x = self.input_proj(x.unsqueeze(-1))           # (batch, seq_len, d_model)
        x = self.transformer(x)          # (batch, seq_len, d_model)
        x = x.flatten(start_dim=1)       # (batch, seq_len * d_model)
        x = self.output_head(x)          # (batch, 1)
        return x.squeeze(-1)             # (batch,)

In [46]:
tr = tree.Tree()
tr.add_node(-1)
tr.add_node(0)

selected_modes = torch.BoolTensor([[1],[1]])

L, q = sequences_oh[0][0].shape
k = torch.zeros(L, q, dtype=dtype, device=device)
Ns0 = energy_models.IndepSites(k)

embedding_size = 4
nhead = 2
transf = TransformerRegressor(q*L, embedding_size, nhead)
nn = energy_models.GenericEnergyModel(transf)

ps = selex_distribution.MultiModeDistribution(nn, normalized=False)
model = selex_distribution.MultiRoundDistribution(Ns0, ps, tr, selected_modes).to(device)

In [47]:
batch_size = 10**3
data_loaders = [data_loading.SelexRoundDataLoader(seq_oh, batch_size=batch_size, device=device) for seq_oh in sequences_oh]
n_rounds = len(data_loaders) 

In [51]:
n_chains = 10**1

chains = training.init_chains(n_rounds, n_chains, L, q, device, dtype)
log_weights = torch.zeros(n_rounds, n_chains, device=device, dtype=dtype)

In [52]:
callbacks = [callback.ConvergenceMetricsCallback(), 
             callback.ParamsCallback(save_every=100), callback.GradientPersistenceCallback()
            ]

In [53]:
n_sweeps = 1
lr = 0.01
max_epochs = 100
weight_decay = 1e-2

training.train(model, data_loaders, total_reads, chains, n_sweeps, lr=lr, l2reg=weight_decay,
               max_epochs=max_epochs, callbacks=callbacks)

 0.00%[                                          ] Epoch: 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
cb_convergence, cb_params, cb_persistence = callbacks

In [ ]:
cb_convergence.plot();

In [ ]:
cb_persistence.plot(figsize=(10,5));

## Epistasis

In [ ]:
filepath = './saved/wildtypes_sites.pkl'
with open(filepath, 'rb') as f:
    wts = pickle.load(f)

wt = wts[experiment_id_idx]
wt_oh = utils.one_hot(wt)

In [ ]:
compute_energy = lambda x: model.selection_energy_at_round(x.to(device).unsqueeze(0), 1).detach()
DDE = utils.epistasis(compute_energy, wt_oh)

In [ ]:
fig, ax = plt.subplots()

im = ax.imshow(selex_dca.get_contact_map(DDE))
# im.set_clim(-0.05, 0.085)
plt.colorbar(im)
ax.set_title('Epistasis')

fig.tight_layout()

In [ ]:
modules = copy.deepcopy(next(nn.model.modules())).cpu()

In [ ]:
first_layer_weights = modules[0].weight.data.reshape(-1, L, q)

In [ ]:
for i in range(first_layer_weights.size(0)):
    logomaker.Logo(pd.DataFrame(first_layer_weights[i], columns=list(utils.TOKENS_DNA)));